In [ ]:
import os
import pandas as pd
from getpass import getpass

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import UpdateColumn

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username=" ",
    password=getpass("DBRepo password: ")
)

print("DBRepo client initialized.")


DBRepo client initialized.


In [ ]:
DB_ID = "    "

In [3]:
#################################################
# 4. Read SQL View Definitions
#################################################

from pathlib import Path

VIEWS_SQL_PATH = Path("../docs/views.sql")

# Check file exists
if not VIEWS_SQL_PATH.exists():

    raise Exception(f"views.sql not found at: {VIEWS_SQL_PATH}")

# Read SQL file
with open(VIEWS_SQL_PATH, "r") as file:

    views_sql = file.read()

print("views.sql loaded successfully.\n")

print(views_sql[:1000])

views.sql loaded successfully.

-- =========================================================
-- views.sql
-- Air Quality Data Stewardship Project
-- TU Wien - WP2 T2.4 View Definitions
-- =========================================================


-- =========================================================
-- Remove existing views if they already exist
-- =========================================================

DROP VIEW IF EXISTS vw_air_quality_features;

DROP VIEW IF EXISTS vw_daily_pollution_summary;

DROP VIEW IF EXISTS vw_station_pollution_summary;



-- =========================================================
-- View: vw_air_quality_features
-- Description:
-- Denormalized ML-ready feature table combining
-- measurements, temporal information, and station metadata.
-- =========================================================

CREATE VIEW vw_air_quality_features AS

SELECT
    m.measurement_id,

    s.station_id,
    s.station_name,
    s.latitude,
    s.longitude,

    t.time

In [16]:
#################################################
# 5. Get Existing Views
#################################################

VIEW_NAME = "vw_air_quality_features"

views = client.get_views(DB_ID)

print("Views returned by get_views:")

for view in views:
    print("-", view.name)

feature_view = None

for view in views:
    if view.name == VIEW_NAME:
        feature_view = view
        break

if feature_view is None:
    raise Exception(
        f"{VIEW_NAME} not found by client.get_views(DB_ID). "
        "Log in again with owner account 12450741, then rerun."
    )

print("\nView found:")
print("Name:", feature_view.name)
#print("ID:", feature_view.id)

Views returned by get_views:
- vw_station_pollution_summary
- vw_daily_pollution_summary
- vw_air_quality_features

View found:
Name: vw_air_quality_features


In [ ]:
#################################################
# 5. Create View: vw_air_quality_features
#################################################

from dbrepo.api.dto import QueryDefinition

# Refresh existing views
existing_views = client.get_views(DB_ID)

existing_view_names = {

    view.name

    for view in existing_views
}

VIEW_NAME = "vw_air_quality_features"

if VIEW_NAME in existing_view_names:

    print(f"{VIEW_NAME} already exists. Skipping.")

else:

    query_definition = QueryDefinition(

        columns=[

            "t_measurement.measurement_id",
            "t_measurement.station_id",
            "t_measurement.time_id",

            "t_measurement.so2",
            "t_measurement.no",
            "t_measurement.no2",
            "t_measurement.co",
            "t_measurement.pm10",
            "t_measurement.o3",
            "t_measurement.pm25",

            "t_measurement.wind_direction",
            "t_measurement.wind_speed",

            "t_measurement.temperature",
            "t_measurement.humidity",
            "t_measurement.pressure",
            "t_measurement.solar_radiation",
            "t_measurement.rain",

            "t_measurement.ben",
            "t_measurement.tol",
            "t_measurement.mxil"
        ],

        datasources=[
            "t_measurement"
        ]
    )

    created_view = client.create_view(

        DB_ID,

        VIEW_NAME,

        query=query_definition,

        is_public=False,

        is_schema_public=False
    )

    print("View created successfully.")

    print("View Name:", created_view.name)

    print("View ID:", created_view.id)

In [14]:
#################################################
# 6. Validate Feature View
#################################################

views = client.get_views(DB_ID)

feature_view = None

for view in views:

    if view.name == "vw_air_quality_features":

        feature_view = view

        break

if feature_view is None:

    raise Exception("View not found.")

#print("View ID:")

#print(feature_view.id)

# Row count
count = client.get_view_data_count(

    DB_ID,

    feature_view.id
)

print("\nRows in view:")

print(count)

# Preview
df_view = client.get_view_data(

    DB_ID,

    feature_view.id,

    page=0,

    size=10
)

display(df_view)


Rows in view:
43800


,ben,co,humidity,measurement_id,mxil,no,no2,o3,pm10,pm25,pressure,rain,so2,solar_radiation,station_id,temperature,time_id,tol,wind_direction,wind_speed
0,0.10,0.29,54.0,1,0.40,2.0,15.0,58.0,5.0,3.0,1009.0,0.0,4.0,17.0,1,14.1,8737,0.60,158.0,1.13
1,0.10,0.26,58.0,3,1.88,1.0,16.0,50.0,7.0,6.0,1007.0,0.0,5.0,17.0,1,13.4,8739,0.92,5.0,0.10
2,0.13,0.24,53.0,4,3.20,1.0,5.0,68.0,6.0,1.0,1006.0,0.0,4.0,17.0,1,15.3,8740,1.23,181.0,0.75
3,0.10,0.26,55.0,5,0.63,2.0,13.0,56.0,6.0,1.0,1006.0,0.0,4.0,17.0,1,14.9,8741,0.50,181.0,0.78
4,0.10,0.25,55.0,2,0.60,1.0,14.0,54.0,4.0,6.0,1008.0,0.0,3.0,17.0,1,14.1,8738,0.90,203.0,0.48
5,0.23,0.63,61.0,8,1.70,52.0,63.0,4.0,19.0,3.0,1006.0,0.0,15.0,17.0,1,12.4,8744,1.85,337.0,0.10
6,0.15,0.42,60.0,7,1.27,33.0,51.0,11.0,9.0,3.0,1006.0,0.0,7.0,17.0,1,13.1,8743,1.20,318.0,0.10
7,0.52,0.61,53.0,10,2.72,67.0,71.0,10.0,22.0,10.0,1006.0,0.0,8.0,28.0,1,14.1,8746,3.15,316.0,0.12
8,0.10,0.30,57.0,6,0.60,4.0,23.0,42.0,8.0,5.0,1006.0,0.0,4.0,17.0,1,14.3,8742,0.73,180.0,0.32
9,0.53,0.64,58.0,9,2.45,87.0,60.0,16.0,18.0,5.0,1005.0,0.0,9.0,17.0,1,12.7,8745,3.08,180.0,1.05


In [ ]:
#################################################
# 7. Create View: vw_daily_pollution_summary
#################################################

from dbrepo.api.dto import QueryDefinition

VIEW_NAME = "vw_daily_pollution_summary"

# Refresh existing views
existing_views = client.get_views(DB_ID)

existing_view_names = {

    view.name

    for view in existing_views
}

if VIEW_NAME in existing_view_names:

    print(f"{VIEW_NAME} already exists. Skipping.")

else:

    query_definition = QueryDefinition(

        columns=[

            "t_measurement.time_id",

            "t_measurement.so2",
            "t_measurement.no",
            "t_measurement.no2",
            "t_measurement.co",
            "t_measurement.pm10",
            "t_measurement.o3",
            "t_measurement.pm25",

            "t_measurement.temperature",
            "t_measurement.humidity",
            "t_measurement.pressure"
        ],

        datasources=[
            "t_measurement"
        ]
    )

    created_view = client.create_view(

        DB_ID,

        VIEW_NAME,

        query=query_definition,

        is_public=False,

        is_schema_public=False
    )

    print("View created successfully.")

    print("View Name:", created_view.name)

    print("View ID:", created_view.id)

In [13]:
#################################################
# 8. Validate View: vw_daily_pollution_summary
#################################################

views = client.get_views(DB_ID)

daily_view = None

for view in views:

    if view.name == "vw_daily_pollution_summary":

        daily_view = view

        break

if daily_view is None:

    raise Exception("View not found.")

#print("View ID:")

#print(daily_view.id)

# Row count
count = client.get_view_data_count(

    DB_ID,

    daily_view.id
)

print("\nRows in view:")

print(count)

# Preview
df_daily_view = client.get_view_data(

    DB_ID,

    daily_view.id,

    page=0,

    size=10
)

display(df_daily_view)


Rows in view:
43800


,co,humidity,no,no2,o3,pm10,pm25,pressure,so2,temperature,time_id
0,0.42,60.0,33.0,51.0,11.0,9.0,3.0,1006.0,7.0,13.1,8743
1,0.61,53.0,67.0,71.0,10.0,22.0,10.0,1006.0,8.0,14.1,8746
2,0.26,58.0,1.0,16.0,50.0,7.0,6.0,1007.0,5.0,13.4,8739
3,0.26,55.0,2.0,13.0,56.0,6.0,1.0,1006.0,4.0,14.9,8741
4,0.25,55.0,1.0,14.0,54.0,4.0,6.0,1008.0,3.0,14.1,8738
5,0.29,54.0,2.0,15.0,58.0,5.0,3.0,1009.0,4.0,14.1,8737
6,0.24,53.0,1.0,5.0,68.0,6.0,1.0,1006.0,4.0,15.3,8740
7,0.64,58.0,87.0,60.0,16.0,18.0,5.0,1005.0,9.0,12.7,8745
8,0.30,57.0,4.0,23.0,42.0,8.0,5.0,1006.0,4.0,14.3,8742
9,0.63,61.0,52.0,63.0,4.0,19.0,3.0,1006.0,15.0,12.4,8744


In [ ]:
#################################################
# 9. Create View: vw_station_pollution_summary
#################################################

from dbrepo.api.dto import QueryDefinition

VIEW_NAME = "vw_station_pollution_summary"

# Refresh existing views
existing_views = client.get_views(DB_ID)

existing_view_names = {

    view.name

    for view in existing_views
}

if VIEW_NAME in existing_view_names:

    print(f"{VIEW_NAME} already exists. Skipping.")

else:

    query_definition = QueryDefinition(

        columns=[

            "t_measurement.station_id",

            "t_measurement.so2",
            "t_measurement.no",
            "t_measurement.no2",
            "t_measurement.co",
            "t_measurement.pm10",
            "t_measurement.o3",
            "t_measurement.pm25",

            "t_measurement.temperature",
            "t_measurement.humidity",
            "t_measurement.pressure",

            "t_measurement.wind_speed",
            "t_measurement.wind_direction"
        ],

        datasources=[
            "t_measurement"
        ]
    )

    created_view = client.create_view(

        DB_ID,

        VIEW_NAME,

        query=query_definition,

        is_public=False,

        is_schema_public=False
    )

    print("View created successfully.")

    print("View Name:", created_view.name)

    print("View ID:", created_view.id)

In [12]:
#################################################
# 10. Validate View: vw_station_pollution_summary
#################################################

views = client.get_views(DB_ID)

station_view = None

for view in views:

    if view.name == "vw_station_pollution_summary":

        station_view = view

        break

if station_view is None:

    raise Exception("View not found.")

#print("View ID:")

#print(station_view.id)

# Row count
count = client.get_view_data_count(

    DB_ID,

    station_view.id
)

print("\nRows in view:")

print(count)

# Preview
df_station_view = client.get_view_data(

    DB_ID,

    station_view.id,

    page=0,

    size=10
)

display(df_station_view)


Rows in view:
43800


,co,humidity,no,no2,o3,pm10,pm25,pressure,so2,station_id,temperature,wind_direction,wind_speed
0,0.25,55.0,1.0,14.0,54.0,4.0,6.0,1008.0,3.0,1,14.1,203.0,0.48
1,0.61,53.0,67.0,71.0,10.0,22.0,10.0,1006.0,8.0,1,14.1,316.0,0.12
2,0.24,53.0,1.0,5.0,68.0,6.0,1.0,1006.0,4.0,1,15.3,181.0,0.75
3,0.30,57.0,4.0,23.0,42.0,8.0,5.0,1006.0,4.0,1,14.3,180.0,0.32
4,0.42,60.0,33.0,51.0,11.0,9.0,3.0,1006.0,7.0,1,13.1,318.0,0.10
5,0.26,55.0,2.0,13.0,56.0,6.0,1.0,1006.0,4.0,1,14.9,181.0,0.78
6,0.29,54.0,2.0,15.0,58.0,5.0,3.0,1009.0,4.0,1,14.1,158.0,1.13
7,0.63,61.0,52.0,63.0,4.0,19.0,3.0,1006.0,15.0,1,12.4,337.0,0.10
8,0.64,58.0,87.0,60.0,16.0,18.0,5.0,1005.0,9.0,1,12.7,180.0,1.05
9,0.26,58.0,1.0,16.0,50.0,7.0,6.0,1007.0,5.0,1,13.4,5.0,0.10
